In [1]:
### Guided Lab:

In [2]:
# here we will go through the process of building a dataset

In [3]:
# training a model to do a task

In [4]:
# Evaluating the model on that task

In [5]:
# Specifically, this toy will be aimed for a helpdesk

In [9]:
# Given a helpdesk message, we must predict a label:
# billing,
# technical,
# account, 
# cancellation

In [10]:
# Step 1. Save the toy dataset

In [11]:
toy = [
    {"id": 1, "text": "I was charged twice this month. Can you refund the extra payment?", "label": "billing"},
    {"id": 2, "text": "My card was declined but my bank says it's fine. What's going on?", "label": "billing"},
    {"id": 3, "text": "The app crashes every time I try to upload a file.", "label": "technical"},
    {"id": 4, "text": "I'm getting a 500 error when I log in from my laptop.", "label": "technical"},
    {"id": 5, "text": "I forgot my password and the reset link never arrives.", "label": "account"},
    {"id": 6, "text": "Please change the email on my account; I no longer have access to the old one.", "label": "account"},
    {"id": 7, "text": "I want to cancel my subscription at the end of the month.", "label": "cancellation"},
    {"id": 8, "text": "Stop renewing my plan. I'm done after this billing cycle.", "label": "cancellation"},
    {"id": 9, "text": "Why did my invoice include an extra seat? I didn't add anyone.", "label": "billing"},
    {"id": 10, "text": "Two-factor authentication isn't working; the code is always invalid.", "label": "account"},
    {"id": 11, "text": "Your website loads but the dashboard is blank.", "label": "technical"},
    {"id": 12, "text": "I need a receipt for last month's payment for my expense report.", "label": "billing"},
    {"id": 13, "text": "Please delete my account and cancel any active plans.", "label": "cancellation"},
    {"id": 14, "text": "I keep getting logged out and asked to sign in again every few minutes.", "label": "technical"},
    {"id": 15, "text": "I can't update my payment method-- it says 'invalid address' but it's correct.", "label": "billing"},
    {"id": 16, "text": "My profile shows the wrong name and I can't edit it.", "label": "account"},
]
labels = ["billing", "technical", "account", "cancellation"]
len(toy)

16

In [12]:
# Now we will split the dataset. Here we will do a simple random split.
# In a real task we would do a more purposeful split (by customer, thread, time, etc)

In [13]:
# the split defines the generalization. Here we have the opportunity to influence
# the models capacity to understand the patterns or perform false positive leakage

In [15]:
# For a common classification, we want the train/val/test split 
# to look like samples from the same distribution we will see in deployment
# we can either preserve class splits (rare vs common) or we do a random split.
# there are many techniques which can be used to make this split tuned to the task
# sometimes one might want the training to be harder than teh real world ddata. 
# we can do techniques that plit by domain (train in arizona, test in antarctica
# time splits, train at time morning, test on time evening
# group split, train on concentrations of a certain amount, test on different concentrations
# avoid deuplication - avoid memorization and ocntamination

In [16]:
# The right split depends on teh question we are trying to answer


In [17]:
# For now we split randomly

In [18]:
import random

def split_dataset(data, train_frac=0.7, seed=0):
    data = list(data)
    random.Random(seed).shuffle(data)
    n_train = int(len(data) * train_frac)
    return data[:n_train], data[n_train:]

train, test = split_dataset(toy, train_frac=0.7, seed=42)
(len(train), len(test), [x["id"] for x in test])

(11, 5, [16, 5, 12, 1, 4])

In [19]:
# 11 are in training set, 5 are in testing set

In [20]:
# Now we build two baselines
# Baseline 1. Majority class

In [21]:
# We count how mamy times each of the class labels appear in the training split
# pick the single most common (frequent label -> the majority class)
# retrun that majority class
# maj_label is that majority label
# This sets a dumb baseline to beat. Can the model alwasy predict the most common class.
# If the model cant beat "alwasy guess the most common outcome"
# then its not learning any useful patterns

In [22]:
from collections import Counter

def majority_baseline(train_data):
    counts = Counter(x["label"] for x in train_data)
    return counts.most_common(1)[0][0]

maj_label = majority_baseline(train)
maj_label

'cancellation'

In [24]:
# So if the model cant guess cancellation as a baseline it is wrong
# the model therefore needs to  be more ocrrect overall then labeling 'cancellation'
# as the correct label for every single item

In [25]:
# Baseline 2

In [26]:
# This second baseline is a bit smarter
# It is considered a rule based classifier baseline.
# it takes text as input, looks for keywords that we specify as associated with each label, 
# returns the label associated with those keywords
# if no keywords are found it returns a default
# we can get specific for the rules via keywords, for example we could make it so a certain 
# number of keywords are needed to hit before a label is given
# If our model cant outperform this then the dataset might be too easy (keywords dominate- then wh
# whats the point of uysing the model? Or the model is not set up correctly)

In [27]:
def keyword_baseline(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ["refund", "invoice", "charged", "payment", "receipt", "billing", "card", "seat"]):
        return "billing"
    if any(w in t for w in ["crash", "error", "500", "bug", "blank", "dashboard", "upload", "website"]):
        return "technical"
    if any(w in t for w in ["password", "reset", "2fa", "two-factor", "email", "profile", "logged out", "login"]):
        return "account"
    if any(w in t for w in ["cancel", "cancellation", "stop renewing", "delete my account", "end of the month"]):
        return "cancellation"
    # fallback
    return "account"

In [28]:
# Now we build an evaluation harness
# Metrics + Confusion Matrix
# The harness is used later for SFT evaluation, LoRA comparisons, RAG system evaluations, judge-model experiments


In [30]:
# Given a labeled test set and any predictor function, we compute standard metrics 
# and confusion matrix to compare systems consistently

In [36]:
# Confusion matrix
# a table that summarizes how a classifiers predictions compare to the true labels
# there are k classes. We create a kxk grid
# the rows are true class, columns are predicted class
# cell i, j = "how many items whose true label is class i are predicted as class j"
# ideally the model predicts a majority of things ocrrect -is less confused - and therefore we see most itms clusted on like
# i and j terms . 'A', 'A'. B,B, and C,C for example below:
# true\pred   A   B   C
# A           8   2   0
# B           1   6   3
# C           0   1   9


In [32]:
def confusion_matrix(y_true, y_pred, labels):
    idx = {lab:i for i,lab in enumerate(labels)}
    m = [[0 for _ in labels] for __ in labels]
    for t,p in zip(y_true, y_pred):
        m[idx[t]][idx[p]] += 1
    return m

In [34]:
# Accuracy is the fraction of examples predicted correctly. This is the total correct, not 
# the same as TP for a class. 
# this represents the overall correctness of teh model (the diagonal of the confusion matrix)
# Answers the question: "how well did the model do at labeling the correct items

In [35]:
def accuracy(y_true, y_pred):
    return sum(t==p for t,p in zip(y_true, y_pred)) / max(1, len(y_true))

In [41]:
# Now we do the per class F1 discussed earlier. This identifies how well the model does at 
# labeling each class correctly. Remember that this metric can be misleading on its own.
# if there is 100 emails 90 percent ham and 10 percent spam and the model says there is 100 ham, it will look like a good
# f1 score for the class ham and a bad f1 for spam. So this metric m,sut be taken in conjunction
# with others. We cannot abse the success of the model from a singel measurement of how well it predicts 
# a class unless we are certain it is truly the only class that matters....

# We also compute the macro_f1 which shows how well the model did -> it is the average of per-class F1 with each class having 
# equal weight. 
# this is the other metric that will show that even though ham was labeled correctly 90% of the time, the model performed 
# poorly (as it takes into consideration how it did with 'spam' 
# this is unweighted in the sense that we sum all F1 and divide the per class F1 metrics by the total number of classes. 

In [42]:
def per_class_f1(y_true, y_pred, labels):
    # micro-utilities
    results = {}
    for lab in labels:
        tp = sum((t==lab and p==lab) for t,p in zip(y_true, y_pred))
        fp = sum((t!=lab and p==lab) for t,p in zip(y_true, y_pred))
        fn = sum((t==lab and p!=lab) for t,p in zip(y_true, y_pred))
        prec = tp / (tp + fp) if (tp+fp) else 0.0
        rec  = tp / (tp + fn) if (tp+fn) else 0.0
        f1   = (2*prec*rec / (prec+rec)) if (prec+rec) else 0.0
        results[lab] = {"precision": prec, "recall": rec, "f1": f1}
    macro_f1 = sum(results[lab]["f1"] for lab in labels) / len(labels)
    return results, macro_f1

In [43]:
# Now we make the evaluation classifer. 

In [44]:
# the Evaluation classifier is designed to:
# extract the ground truth labels from the training set
# run the predictor (the model) on the training set
# compute the accuracy of the model (the matrix we created earlier) identify how many the model got correct
# Run the F1 per class and macro F1
# Returns a dict containing:
# the accuracy, the macro_f1
# per_class metrics
# confusion matrix
# and raw y_true and y_prediciton so we can inspect individual mistakes as they appear in totality (via confusion matrix)


In [45]:
# this classifier is what is run during evaluation. We run this classifer on the held out set. 
# We can run it on validation set (which we dont ahve for this toy data set) 
# to inform tuning of the model

# once we are happy with the models performance ont e vlaidation set we then run it once on the test set foor the final
# unbiased, result

In [47]:
def evaluate_classifier(test_data, predict_fn, labels):
    y_true = [x["label"] for x in test_data]
    y_pred = [predict_fn(x["text"]) for x in test_data]
    acc = accuracy(y_true, y_pred)
    pc, macro = per_class_f1(y_true, y_pred, labels)
    cm = confusion_matrix(y_true, y_pred, labels)
    return {"accuracy": acc, "macro_f1": macro, "per_class": pc,
            "confusion": cm, "y_true": y_true, "y_pred": y_pred}

In [48]:
# Now we run the system. 
# we print the keyword and the majority baseline accuracy and ther keyword macro F1
# this tells us our baseline metrics to beat. "Can our mdoel beat the accuracy and F1 metrics of the 
# previously defined majority label?
# Can it beat the keyword -> label predictor we built?

In [49]:
# Majority baseline predictor
def predict_majority(_text):
    return maj_label

maj_results = evaluate_classifier(test, predict_majority, labels)
kw_results  = evaluate_classifier(test, keyword_baseline, labels)

maj_results["accuracy"], kw_results["accuracy"], kw_results["macro_f1"]

(0.0, 1.0, 0.75)

In [52]:
# majority basleine did terribly! none fo the majority label happened to be in our test set
# the keyword baseline did incredibly. all keywords appeared in the test set and were appropriately linked
# to their respective labels
# macro f1 being 0.75 is interesting. Becasue F1 takes into consideration ALL labels possible, this F1 indicates that 
# a label in Labels had F1=0 under the per class calcualtion in the test set. That is often possible 
# there is at least one label possible (in the training set) that does not appear in the test set. 
# we can confirm this by counting test labels

In [53]:
from collections import Counter
Counter(x["label"] for x in test), labels

(Counter({'account': 2, 'billing': 2, 'technical': 1}),
 ['billing', 'technical', 'account', 'cancellation'])

In [54]:
# this explains it. Cancellation was our majority label baseline (and it did not appear in 
# the test set according to this. 
# additioanlly, becasue there is 5 labels total and only 4 appeared in the test set (no cancellation), this explains 
# why the keyword macro f1 results indicated less than one (meaning that the cancellation label received an f1 of 0 in the 
# test set, 0 predicted cancellation labels, 0 actual cancellation true labels)

In [57]:
# the keyword baseline can still get accuracy of 1.0 by labeling the 5 test examples that appear in
# the test set correctly, but macro F1 averages across all 4 labels that exist and because F1 = 0 for the 'cancellation'
# label in the test set (as it wasnt one of the 5 labels present in the test set) , the macro f1 is (1 + 1 + 1 + 0) / 4 = 0.75

In [58]:
# So now we fundamnetally have our baselines to beat by the model.
# we could of course tune and iterate, but for this small dataset toy, this will suffice

In [59]:
# we can print the baseline metrics as follows via confusion metrics if we want to see this visually:

In [60]:
def print_confusion(cm, labels):
    header = "true\\pred".ljust(12) + " ".join(lab.ljust(12) for lab in labels)
    print(header)
    for i,lab in enumerate(labels):
        row = lab.ljust(12) + " ".join(str(cm[i][j]).ljust(12) for j in range(len(labels)))
        print(row)

print("Majority confusion:")
print_confusion(maj_results["confusion"], labels)

print("\nKeyword confusion:")
print_confusion(kw_results["confusion"], labels)

Majority confusion:
true\pred   billing      technical    account      cancellation
billing     0            0            0            2           
technical   0            0            0            1           
account     0            0            0            2           
cancellation0            0            0            0           

Keyword confusion:
true\pred   billing      technical    account      cancellation
billing     2            0            0            0           
technical   0            1            0            0           
account     0            0            2            0           
cancellation0            0            0            0           


In [61]:
# For majority, the label predicted is 'cancellation', therefore
# given that cancellation was not in the test set, it
# is false for every prediction. if we go by the majority label, then it was incorrect 5/5
# times. For keywords, the keywords in the test set correctly found their actual labels 5 out
#  of t5 times, becasue cancellation was not part of the test set, we see accurately it was not
# predicted at all


In [62]:
# Now we will evaluate Nemotron nano 3 on the test set. 
# We must evaluate with temperature as close to 0 as possible given our goal is repeatable determinism,
# We also need to ensure a consistent prompt tempalte
# Ensure a strict output format


In [64]:
## Prompt Template
# ideally JSON which forces a single output label
# We start with a system message to prime the model

In [65]:
# A system message is the highest priority instructions that set the models role/behaviour
# highest priority protocol to follow
# " You are a routing classifier. Output only JSON "

In [67]:
# *****IMPORTANT*****
# Chat tuned models are trained on text with labels on text data scuh as system message, user message, developer message 
# it is this instruction tuning / RLHF of the model that allows them to learn the significance/ prioritizzation of one vs the other.

# It is still all tokens though, the messages are converted to tokens just as anyu prompt would be. 
# the tempalte we give the model might induce special
# characters <|im_start|>system, <|im_start|>user, <|im_start|>assistant, <|im_end|>
# and it is this that the model can associate with its training patterns which then infleunce its 
# encoding and therefore how it answers the prompt upon decode
# Take away, these USer message, ssytem message, or dev message are jsut otkens like any other.
# BUT models are trained on these labels during their trusction tuning
# therefore they are encoded patternistically and infleucne the outputs. 
# One could achieve simialr results in terms of prioritization if they made dog-waterbottle-pooter higher prioritization 
# than weezle-beedle-doog
# the model would learn that pattern and if that is the template the developers popublish alognside theri model it would achieve
# the same result

In [71]:
# lets load nemotron and inspect its chat template:

In [70]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
revision = "2e43387afd60157064e5bef4e9a583f887c6dfdd"  # your cached snapshot
cache_dir = "/data/hf/hub"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=True,
    revision=revision,
    cache_dir=cache_dir,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    revision=revision,
    cache_dir=cache_dir,
    local_files_only=True,
)

print("Loaded tokenizer + model on", model.device)

/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

Loaded tokenizer + model on cuda:0


In [72]:
print(tokenizer.chat_template)

{% macro render_extra_keys(json_dict, handled_keys) %}
    {%- if json_dict is mapping %}
        {%- for json_key in json_dict if json_key not in handled_keys %}
            {%- if json_dict[json_key] is mapping or (json_dict[json_key] is sequence and json_dict[json_key] is not string) %}
                {{- '\n<' ~ json_key ~ '>' ~ (json_dict[json_key] | tojson | safe) ~ '</' ~ json_key ~ '>' }}
            {%- else %}
                {{-'\n<' ~ json_key ~ '>' ~ (json_dict[json_key] | string) ~ '</' ~ json_key ~ '>' }}
            {%- endif %}
        {%- endfor %}
    {%- endif %}
{% endmacro %}
{%- set enable_thinking = enable_thinking if enable_thinking is defined else True %}
{%- set truncate_history_thinking = truncate_history_thinking if truncate_history_thinking is defined else True %}

{%- set ns = namespace(last_user_idx = -1) %}
{%- set loop_messages = messages %}
{%- for m in loop_messages %}
  {%- if m["role"] == "user" %}
    {%- set ns.last_user_idx = loop.index0 %}
  {

In [73]:
# this is the pormpt compiler for nemotron chat. Nemotron was tuned 
# in conjunction with this tempalte and therefore this template influences its pattern associations 
# and therefore behaviour (how it predicts the next token)

In [74]:
# For nemotron we msut adhere to this tempalte if we want good results. 
# We use tokenizer.apply_chat_tempalte(...) so the prompt includes the markers Nemotron was tuned on
#  <|im_start|>system ... <|im_end|>
#  <|im_start|>user ... <|im_end|>
#  <|im_start|>assistant ... <|im_end|>

In [75]:
# we decide if we want thinking via enable_thinking=True or False, and that 'invites' the model to 
# produce a reasoning segment.
# as an example fundamentally. If we pass a message:

In [78]:
# message = [{"role": "user", "content": "Hello}]

# and we call: tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, ...)

# the tokenizer uses nemotrons chat_tempalte.jinja to automatically insert the special token : 
# (<|im_start|>user, <|im_end|>, <|im_start|>assistant, etc.) that nemotron was trained on dealing with


In [79]:
# so now back to our task. Our system message : "You are a routing classifier. Output only JSON."
# our user message: "Given the helpdesk message, classify into one of: billing, technical, account,
# cancellation.

In [80]:
# We can also make it loop through all the messages in our training set